## Chargement des données

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd
import os

In [2]:
import sys
sys.path.append("../pipelines")
from aq import *

print("Lot :", BATCH, "| date de référence :", DATE_REF)
print(lire("SELECT VERSION() AS version, @@collation_server AS collation"))

Lot : L-2026-10-01 | date de référence : 2026-10-01
                  version          collation
0  11.2.6-MariaDB-ubu2204  utf8mb4_nopad_bin


## US-01

In [3]:
# US-01 — Inventaire des sources cif et crm

# 1.1 Tables, colonnes et clés primaires
cols = lire("""
    SELECT table_schema AS base, table_name AS `table`, COUNT(*) AS colonnes
    FROM information_schema.columns
    WHERE table_schema IN ('cif', 'crm')
    GROUP BY table_schema, table_name
""")

cles = lire("""
    SELECT table_schema AS base, table_name AS `table`,
           GROUP_CONCAT(column_name ORDER BY ordinal_position) AS cle_primaire
    FROM information_schema.key_column_usage
    WHERE table_schema IN ('cif', 'crm') AND constraint_name = 'PRIMARY'
    GROUP BY table_schema, table_name
""")

inventaire = cols.merge(cles, how="left", on=["base", "table"])

# 1.2 Comptage exact des lignes
inventaire["lignes"] = [
    int(lire(f"SELECT COUNT(*) AS n FROM {b}.{t}").n[0])
    for b, t in zip(inventaire.base, inventaire.table)
]

print(inventaire.sort_values(["base", "table"]).to_string(index=False))

# 1.3 Tables sans clé primaire
print(
    "\nTables sans clé primaire :",
    inventaire[inventaire.cle_primaire.isna()][["base", "table"]].values.tolist()
)

base        table  colonnes cle_primaire  lignes
 cif      clients        10      column0   40000
 cif       compte         4      column0   45000
 cif         pays         2    code_pays     250
 crm       client        13    client_id  100000
 crm      contrat         9   contrat_id  100000
 crm     sinistre        10  sinistre_id  100000
 crm type_contrat         3    code_type      12

Tables sans clé primaire : []


## US-02

In [4]:
# US-02
tables_bronze = lire("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'bronze'
    ORDER BY table_name
""")

print(tables_bronze.to_string(index=False))

      table_name
     cif_clients
      cif_compte
        cif_pays
      crm_client
     crm_contrat
    crm_sinistre
crm_type_contrat


In [5]:
%run ../pipelines/01_ingest_bronze.py

In [6]:
for table in tables_bronze["table_name"]:
    resultat = lire(f"""
        SELECT COUNT(*) AS nb_lignes
        FROM bronze.{table}
        WHERE _batch_id = :batch
    """, batch=BATCH)

    print(f"{table} : {resultat['nb_lignes'].iloc[0]} lignes")

cif_clients : 40000 lignes
cif_compte : 45000 lignes
cif_pays : 250 lignes
crm_client : 100000 lignes
crm_contrat : 100000 lignes
crm_sinistre : 100000 lignes
crm_type_contrat : 12 lignes


In [7]:
lire("""
    SELECT COUNT(*) AS nb_lignes,
           SUM(_ingested_at IS NULL) AS dates_manquantes
    FROM bronze.crm_client
    WHERE _batch_id = :batch
""", batch=BATCH)

,nb_lignes,dates_manquantes
0,100000,0


In [19]:
# US-02 — Vérification d'idempotence : comptage avant relance

avant = lire("""
    SELECT COUNT(*) AS nb
    FROM bronze.crm_client
    WHERE _batch_id = :b
""", b=BATCH)

print("Avant :", avant.iloc[0]["nb"])

Avant : 100000


In [20]:
# US-02 — Vérification d'idempotence : relance du même lot

%run ../pipelines/01_ingest_bronze.py

apres = lire("""
    SELECT COUNT(*) AS nb
    FROM bronze.crm_client
    WHERE _batch_id = :b
""", b=BATCH)

print("Après :", apres.iloc[0]["nb"])
print("Écart :", apres.iloc[0]["nb"] - avant.iloc[0]["nb"])

Après : 100000
Écart : 0


## US-03

In [8]:
# US-03 : Porte B — Réconciliation source → bronze (RQ-021)

def somme_controle(base, table, ou=""):
    colonnes = lire("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :s
          AND table_name = :t
          AND column_name NOT LIKE '\\_%'
          AND column_name <> 'bronze_row_id'
        ORDER BY ordinal_position
    """, s=base, t=table).column_name.tolist()

    liste = ", ".join(colonnes)

    r = lire(f"""
        SELECT COUNT(*) AS n,
               COALESCE(SUM(CRC32(CONCAT_WS('|', {liste}))), 0) AS crc
        FROM {base}.{table} {ou}
    """)

    return int(r.n[0]), int(r.crc[0])

In [9]:
somme_controle("crm", "client")
somme_controle("bronze", "crm_client", f"WHERE _batch_id = '{BATCH}'")

(100000, 214577921396822)

In [10]:
print("Source :", somme_controle("crm", "client"))
print("Bronze :", somme_controle("bronze", "crm_client", f"WHERE _batch_id = '{BATCH}'"))

Source : (100000, 214577921396822)
Bronze : (100000, 214577921396822)


In [11]:
SOURCES = [("crm", "client", "crm_client"), ("crm", "contrat", "crm_contrat"), ("crm", "type_contrat", "crm_type_contrat"), ("crm", "sinistre", "crm_sinistre"),
           ("cif", "clients", "cif_clients"), ("cif", "compte", "cif_compte"), ("cif", "pays", "cif_pays")]

In [12]:
porte_b = []
for base, table, cible in SOURCES:
    n_src, crc_src = somme_controle(base, table)
    n_brz, crc_brz = somme_controle("bronze", cible, f"WHERE _batch_id = '{BATCH}'")
    ecarts = abs(n_src - n_brz) + int(crc_src != crc_brz)
    porte_b.append(mesurer("BRONZE", "RQ-021", cible, n_src, ecarts))
    # RQ-022 : chaque ligne porte lot, source, horodatage et empreinte
    meta = lire(f"""SELECT COUNT(*) AS n, COALESCE(SUM(_batch_id IS NULL OR _batch_id = '' OR _source_system IS NULL OR _source_system = ''
               OR _ingested_at IS NULL OR _row_hash IS NULL OR _row_hash = ''), 0) AS ko FROM bronze.{cible} WHERE _batch_id = '{BATCH}'""")
    porte_b.append(mesurer("BRONZE", "RQ-022", cible, int(meta.n[0]), int(meta.ko[0])))
print(pd.DataFrame(porte_b).sort_values(["rule_id", "table"]).to_string(index=False))   # aucun écart attendu

rule_id couche            table  testees  echecs  conformite_pct
 RQ-021 BRONZE      cif_clients    40000       0           100.0
 RQ-021 BRONZE       cif_compte    45000       0           100.0
 RQ-021 BRONZE         cif_pays      250       0           100.0
 RQ-021 BRONZE       crm_client   100000       0           100.0
 RQ-021 BRONZE      crm_contrat   100000       0           100.0
 RQ-021 BRONZE     crm_sinistre   100000       0           100.0
 RQ-021 BRONZE crm_type_contrat       12       0           100.0
 RQ-022 BRONZE      cif_clients    40000       0           100.0
 RQ-022 BRONZE       cif_compte    45000       0           100.0
 RQ-022 BRONZE         cif_pays      250       0           100.0
 RQ-022 BRONZE       crm_client   100000       0           100.0
 RQ-022 BRONZE      crm_contrat   100000       0           100.0
 RQ-022 BRONZE     crm_sinistre   100000       0           100.0
 RQ-022 BRONZE crm_type_contrat       12       0           100.0


In [13]:
# Épreuve de l'écart volontaire : sur une COPIE, jamais sur bronze elle-même, et sans écrire le résultat dans dq.dq_history
executer("DROP TABLE IF EXISTS bronze.crm_client_test")
executer("CREATE TABLE bronze.crm_client_test LIKE bronze.crm_client")
executer("INSERT INTO bronze.crm_client_test SELECT * FROM bronze.crm_client")
# À vous : altérez la copie (un UPDATE ... LIMIT ... ou une suppression, un doublon inséré), par exemple :
executer("DELETE FROM bronze.crm_client_test LIMIT 1")
n_t, crc_t = somme_controle("bronze", "crm_client_test")
n_s, crc_s = somme_controle("crm", "client")
print("copie contre source : écarts =", abs(n_s - n_t) + int(crc_s != crc_t), "(0 tant que la copie n'est pas altérée)")
executer("DROP TABLE bronze.crm_client_test")

copie contre source : écarts = 2 (0 tant que la copie n'est pas altérée)


In [18]:
# US-03 — Vérification de l'historique des mesures RQ-021 et RQ-022

lire("SELECT COUNT(*) AS nb_mesures FROM dq.dq_history")

,nb_mesures
0,14


## US-06 — Profilage complet de bronze.crm_client

In [21]:
# US-06 — Étape 1 : Charger les clients de Bronze
# Adaptation du workflow du professeur : bronze.crm_client au lieu de crm.client

df = lire("SELECT * FROM bronze.crm_client WHERE _batch_id = :b", b=BATCH)

print(f"[OK] {len(df)} lignes chargées")

[OK] 100000 lignes chargées


In [22]:
# US-06 — Étape 2 : Génération du rapport YData-Profiling

from ydata_profiling import ProfileReport

# Profilage complet de la table crm.client (100 000 lignes)
profile = ProfileReport(
    df,
    title="AssurConnect — Profilage crm.client (100 000 lignes)",
    explorative=True
)
profile.to_file("/home/jovyan/work/reports/ydata_profiling_client.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 18/18 [00:13<00:00,  1.30it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
# US-06 — Étape 4 : Compter les codes postaux français à 4 chiffres (A01)

print(
    "Codes postaux FR à 4 chiffres (A01) :",
    ((df["cp"].str.len() == 4) & (df["pays"] == "FR")).sum()
)

Codes postaux FR à 4 chiffres (A01) : 1456


In [25]:
# US-06 — Étape 5 : Vérification des codes pays (A02)
# Code du professeur, avec la connexion existante lire()

ref_pays = lire("SELECT code_pays FROM silver.ref_pays")

print(
    "Pays hors référentiel (A02, A18) :",
    (~df["pays"].isin(ref_pays["code_pays"])).sum()
)

Pays hors référentiel (A02, A18) : 3050


In [26]:
# US-06 — Étape 6 : Souscripteurs sans téléphone (A03)

sans_tel = df.loc[df["type_personne"] == "souscripteur", "telephone"].fillna("").str.strip() == ""
print("Souscripteurs sans téléphone (A03) :", sans_tel.sum())

Souscripteurs sans téléphone (A03) : 5989


In [27]:
# US-06 — Étape 7 : E-mails déjà vus sur un autre client (A05)

print("E-mails déjà vus sur un autre client (A05) :", df["email"].dropna().duplicated().sum())

E-mails déjà vus sur un autre client (A05) : 2840


In [28]:
# US-06 — Étape 8 : Doublons stricts nom + prénom + naissance (A15)

print("Doublons stricts nom + prénom + naissance (A15) :", df.dropna(subset=["nom", "prenom", "date_naissance"]).duplicated(subset=["nom", "prenom", "date_naissance"]).sum())

Doublons stricts nom + prénom + naissance (A15) : 738


In [29]:
# US-06 — Étape 9 : Retrouver le cas Meunier

meunier = df[df["client_id"].isin(["A-4471", "H-9012", "S-2208"])]
meunier[["client_id", "raison_sociale", "adresse", "cp", "ville", "telephone"]]

,client_id,raison_sociale,adresse,cp,ville,telephone
34449,A-4471,Société Meunier SARL,27 rue Pasteur,69003,Lyon,0478473441
73792,H-9012,MEUNIER SARL,27 rue Pasteur,69003,Lyon,0409980316
88068,S-2208,Sté Meunier S.A.R.L.,27 rue Pasteur,69003,Lyon,0417379714
